In [1]:
import pandas as pd
from pathlib import Path
from data_preparation import load_graphs_from_db, load_sequences_from_db, \
                              load_metadata_from_db, describe_corpus
from fsm_recipes import RecipeFSM, cluster_recipes_from_fsm, describe_clusters
from mba_sequential import PrefixSpanSPM
from run_pipeline_fsm_mba import run_pipeline

In [2]:
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent   
DATA_DIR = PROJECT_ROOT / "data"

In [3]:


DB = DATA_DIR/"recipe_graphs.db"
limit = 10000
graphs    = load_graphs_from_db(DB, limit=limit)
sequences = load_sequences_from_db(DB, limit= limit)
metadata  = load_metadata_from_db(DB, limit=limit)
support = 0.01
describe_corpus(graphs, sequences, metadata)

# FSM
fsm = RecipeFSM(min_support=support, min_edge_weight=4)
fsm.fit(graphs)
print(fsm.to_dataframe().head(20))

# Clustering
assignments = cluster_recipes_from_fsm(fsm, method="kmeans")
print(describe_clusters(assignments, metadata, fsm))

# MBA séquentiel
spm = PrefixSpanSPM(min_support=support)
spm.fit(list(sequences.values()))
print(spm.to_dataframe().head(20))
print("Gestes déclencheurs :", spm.trigger_gestures(5))

# Matrice de features pour GNN
matrix, recipe_ids, pattern_labels = fsm.vectorize_recipes()
# → matrix.shape = (n_recettes, n_patterns)
# → utilisable directement comme features dans PyTorch Geometric

[Chargement] 9639 graphes chargés depuis C:\Users\nguem\Downloads\recipes-20250605T021334Z-1-002\recipes\data\recipe_graphs.db
[Séquences] 9561 séquences extraites depuis C:\Users\nguem\Downloads\recipes-20250605T021334Z-1-002\recipes\data\recipe_graphs.db

--- Statistiques corpus ---
  Recettes              : 9639
  Gestes uniques        : 195
  Longueur moy. séq.    : 4.62
  Nœuds moy. graphe     : 6.02
  Arêtes moy. graphe    : 7.02
  Top gestes  : ['add', 'stir', 'mix', 'cut', 'combine', 'serve', 'place', 'pour', 'drain', 'remove']
  Top transitions : [('add→stir', 3720), ('stir→add', 2696), ('add→mix', 1958), ('mix→add', 1506), ('saute→add', 1199)]
  Recettes avec cycles  : 6628 (66.3%)
[FSM] 9639 graphes  min_support=0.01 (96 recettes min)  min_edge_weight=4
[FSM] 743 arêtes distinctes dans le corpus
[FSM] 3 arêtes fréquentes (longueur 1)
[FSM] Total : 3 patterns fréquents
      pattern  length  support  n_recipes
0  add → stir       1   0.0284        274
1  stir → add       1   

c:\Users\nguem\Downloads\forage de donnees\devoir1\.conda\lib\site-packages\sklearn\base.py:1152: ConvergenceWarning: Number of distinct clusters (8) found smaller than n_clusters (24). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


[Clustering] 8 clusters  (0 recettes non assignées)
   cluster  n_recettes                         top_patterns  \
0        0        9220                                        
1        1         143                           add → stir   
2        2         124              add → stir | stir → add   
3        3          91                            add → mix   
4        4          49                           stir → add   
5        5           5               stir → add | add → mix   
6        6           4               add → stir | add → mix   
7        7           3  add → stir | stir → add | add → mix   

                                            exemples  
0  Worlds Best Mac and Cheese | Dilly Macaroni Sa...  
1  Cauliflower "Mac & Cheese" Bake | Peanut Butte...  
2  Walnut and Rice Loaf With Chive Sauce | Spinat...  
3  Carrot Cake for Careful People | Four Alarm Ch...  
4  Paella | Back To Basics: Red Beans And Rice | ...  
5  Easy coconut slice | Mushroom Pineapple Rice |.

In [5]:
fsm.to_dataframe().head(20)

,pattern,length,support,n_recipes
0,add → stir,1,0.1123,1082
1,stir → add,1,0.0846,815


In [6]:
spm.to_dataframe().head(20)

,antecedent,consequent,support,confidence,lift,n_recipes
0,add,serve,0.0807,0.1617,0.8461,765
1,add,stir,0.1311,0.2627,0.7572,1244


In [14]:
recipe = pd.read_csv(DATA_DIR / "recipes.csv")
ingredients = pd.read_csv(DATA_DIR / "recipe_ingredients.csv")


In [19]:
ingredients = ingredients.groupby("id")["ingredient"] \
    .agg(lambda x: " ".join(x.dropna().astype(str))) \
    .reset_index()

In [21]:
ingredients = ingredients.merge(recipe[["id","title"]],on="id")

In [22]:
ingredients.head(16)

,id,ingredient,title
0,000018c8a5,6 ounces penne 2 cups Beechers Flagship Cheese...,Worlds Best Mac and Cheese
1,000033e39b,1 c. elbow macaroni 1 c. cubed American cheese...,Dilly Macaroni Salad Recipe
2,000035f7ed,"8 tomatoes, quartered Kosher salt 1 red onion,...",Gazpacho
3,00003a70b1,2 12 cups milk 1 12 cups water 14 cup butter m...,Crunchy Onion Potato Bake
4,00004320bb,1 (3 ounce) package watermelon gelatin 14 cup ...,Cool 'n Easy Creamy Watermelon Pie
5,0000631d90,12 cup shredded coconut 1 lb lean ground beef ...,Easy Tropical Beef Skillet
6,000075604a,2 Chicken thighs 2 tsp Kombu tea 1 White pepper,Kombu Tea Grilled Chicken Thigh
7,00007bfd16,"6 -8 cups fresh rhubarb, or 6 -8 cups frozen r...",Strawberry Rhubarb Dump Cake
8,000095fc1d,"8 ounces, weight Light Fat Free Vanilla Yogurt...",Yogurt Parfaits
9,0000973574,2 cups flour 1 tablespoon cinnamon 2 teaspoons...,Zucchini Nut Bread


In [23]:
ingredients.to_csv(DATA_DIR / "ingredients_with_titles.csv", index=False)

In [ ]:
from results_persistence import load_results

results = load_results("graph_mining_results")

# Accès direct à chaque résultat
df_patterns  = results["patterns"]        # DataFrame pandas
df_spm       = results["spm_rules"]       # DataFrame pandas
triggers     = results["triggers"]         # liste de dicts
assignments  = results["assignments"]      # { recipe_id: cluster_id }

# Matrice prête pour PyTorch Geometric
matrix       = results["matrix"]           # np.ndarray (n_recettes × n_patterns)
recipe_ids   = results["recipe_ids"]
pattern_labels = results["pattern_labels"]

['Cheesy Herbed Egg Sandwich',
 'Curried Pumpkin and Smoked Salmon Soup',
 'Apples And Almonds In Flaky Pastry Recipe',
 'Balsamic Chicken Pasta with Fresh Cheese',
 'Apple-Currant Bars',
 'Chirashi (Scattered) Sushi',
 'Dinner In A Skillet Recipe',
 'Super Easy Pie Crust',
 'Old-Fashioned Sweet Potato Pie',
 'Mummy Cupcakes',
 'Butternut Squash Soup or Bisque (Roasting Method)',
 'Left over Make over Turkey Pot Pie',
 'Whole Wheat Waffles',
 'Hot Corned Beef Buns',
 'Creme Curd Cups',
 'Barbecue Baked Potatoes',
 'Crab Salad with Endive and Tomato-Cilantro Sauce',
 'Anise Biscotti',
 "Blondie's Walnut Balls",
 'Peach Scones',
 'Frosty Strawberry Squares',
 'Peanut Butter Bread',
 'Salt Free, Low Cholesterol Sugar Cookies Recipe',
 'Spoonriver Melon Mint Soup',
 'Revoltillos',
 'Wonderful and Good For Health Fruit Salad',
 'Curry Wheat Berry Salad',
 'Half-Time Sunshine Bars',
 "Grandma's Caramel Corn",
 'Mango Atchar',
 'Caramelized Beef Skewers',
 'Shalom Bayit Kugel (Potato Kugel)',

In [13]:
len(recipe)

1029720